In [1]:
import sys
sys.path.insert(0, "/content/llm")

import os
import datetime
import tensorflow as tf
from tensorflow.python.keras.engine import training
from preprocess.processing_pipeline import ProcessingPipeline
from model.transformer import Transformer
from optimizer_schedule import VariableLR
from datasets import load_dataset
from dotenv import load_dotenv

I0000 00:00:1795513592.923715   84598 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795513592.967464   84598 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795513594.109307   84598 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/rayane/.python_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
%load_ext tensorboard

In [7]:
load_dotenv()

True

In [8]:
dataset = load_dataset(
        "parquet",
        data_files={
            "train": os.path.join(os.environ["HF_HOME"],
            "fineweb/datasets--HuggingFaceFW--fineweb/snapshots/9bb295ddab0e05d785b879661af7260fed5140fc/sample/10BT/*.parquet"),
        },
        split="train",
        streaming=True)

pipeline = ProcessingPipeline(dataset=dataset)
train_datas = pipeline.process_dataset()

E0000 00:00:1795513638.363814   84598 cuda_platform.cc:57] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
=== Source Location Trace: === 
external/xla/xla/stream_executor/cuda/cuda_status.cc:50



In [9]:
transformer = Transformer(vocab_size=len(pipeline.tokenizer), dk=32, dv=32, d_model=128, h=4, N=4)

enc_in = tf.keras.layers.Input(shape=(None,))
dec_in = tf.keras.layers.Input(shape=(None,))
enc_out = transformer((enc_in, dec_in))

model = tf.keras.models.Model(inputs=[enc_in, dec_in], outputs=enc_out)
print(model.summary(expand_nested=True))

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer         │ (None, None,      │ 24,352,594 │ input_layer[0][0… │
│ (Transformer)       │ 50258)            │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ casual_mask    │ ?                 │          0 │ -                 │
│ (CasualMask)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ encoder        │ ?                 │  8,803,072 │ -                 │
│ (Encoder)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ ?                 │          0 │ -                 │
│ positional_embeddi… │                   │            │                   │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ ?                 │    592,512 │ -                 │
│ encoder_layer       │                   │            │                   │
│ (EncoderLayer)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ ?                 │    592,512 │ -                 │
│ encoder_layer_1     │                   │            │                   │
│ (EncoderLayer)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ ?                 │    592,512 │ -                 │
│ encoder_layer_2     │                   │            │                   │
│ (EncoderLayer)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └             │ ?                 │    592,512 │ -                 │
│ encoder_layer_3     │                   │            │                   │
│ (EncoderLayer)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └ embedding   │ (None, None, 128) │  6,433,024 │ -                 │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ decoder        │ ?                 │ 15,549,522 │ -                 │
│ (Decoder)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └ casual_mask │ ?                 │          0 │ -                 │
│ (CasualMask)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └ dense_60    │ (None, None,      │  6,483,282 │ -                 │
│ (Dense)             │ 50258)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│       └ embedding_1 │ (None, None, 128) │  6,433,024 │ -                 │
│ (Embedding)         │                   │            │                 

 Total params: 24,352,594 (92.90 MB)

 Trainable params: 24,352,594 (92.90 MB)

 Non-trainable params: 0 (0.00 B)

None


In [10]:
EPOCHS = 100
CHECKPOINT_EVERY = 100
LOG_EVERY = 100
step_per_epoch = 16
total_step = EPOCHS * step_per_epoch

In [12]:
loss_func = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction="none")
scheduler = VariableLR(d_model=512, warmup_steps=4000)

optimizer = tf.keras.optimizers.Adam(
    learning_rate=scheduler,
    beta_1=0.9,
    beta_2=0.98,
    epsilon=10e-09)

def loss_fn(y_true, y_pred):

    loss = loss_func(y_true, y_pred)
    mask = tf.cast(
        tf.not_equal(y_true, pipeline.tokenizer.pad_token_id),
        loss.dtype)

    loss *= mask

    return tf.reduce_sum(loss) / tf.maximum(tf.reduce_sum(mask), 1.0)

In [13]:
@tf.function
def train_step(enc_input, dec_input, target):

    with tf.GradientTape() as tape:

        prediction = model((enc_input, dec_input), training=True)
        loss = loss_fn(target, prediction)

    gradient = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradient, model.trainable_variables))
    return loss

In [14]:
@tf.function
def validation_step(enc_input, dec_input, target):

    prediction = model((enc_input, dec_input), training=False)
    return loss_fn(target, prediction)

In [15]:
checkpoint_dir = "checkpoints"

checkpoint = tf.train.Checkpoint(
    model=model,
    optimizer=optimizer,
    iterator=iter(train_datas.repeat())
)

checkpoint_manager = tf.train.CheckpointManager(
    checkpoint,
    checkpoint_dir,
    max_to_keep=5,)

optimizer.build(model.trainable_variables)

In [16]:
log_dir = os.path.join(
    "logs", "transformer", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))

summary_wrter = tf.summary.create_file_writer(log_dir)

In [17]:
try:

    for epoch in range(EPOCHS):

        train_loss_sum = 0.0
        train_token_count = 0

        for (enc_in, dec_in), target in train_datas:

            loss = train_step(enc_in, dec_in, target)
            valid_token = tf.reduce_sum(tf.cast(
                tf.not_equal(target, pipeline.tokenizer.pad_token_id),
                tf.float32))

            train_loss_sum += tf.cast(loss, tf.float32) * tf.cast(valid_token, tf.float32)
            train_token_count += tf.cast(valid_token, tf.int32)
            step = int(optimizer.iterations.numpy())

            if step % LOG_EVERY == 0:

                with summary_wrter.as_default():

                    tf.summary.scalar("Training/Loss", loss, step=step)
                    tf.summary.scalar("Training/Learning_Rate", optimizer.learning_rate, step=step)
                    print(f"Epoch: {epoch} | Step: {step} | Loss: {loss}")

            if step % CHECKPOINT_EVERY == 0:
                checkpoint_manager.save(checkpoint_number=step)
                summary_wrter.flush()

except KeyboardInterrupt:
    print("Cancelling training.")

finally:
    checkpoint_manager.save(checkpoint_number=int(optimizer.iterations.numpy()))
    summary_wrter.flush()

/home/rayane/.python_env/lib/python3.14/site-packages/keras/src/backend/tensorflow/ops/nn.py:1524: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(
W0000 00:00:1795513710.488887   84702 cpu_allocator_impl.cc:174] Allocation of 411713536 exceeds 10% of free system memory.
W0000 00:00:1795513713.125474   84598 prefetch_dataset_op.cc:476] SEVERE STARVATION: Element UID 1795513699457252087 in buffer 'PrefetchDataset/_12' sat unconsumed for 13.5759 seconds!
W0000 00:00:1795513713.467948   84703 cpu_allocator_impl.cc:174] Allocation of 411713536 exceeds 10% of free system memory.
W0000 00:00:1795513722.551668   84703 cpu_allocator_impl.cc:174] Allocation of 411713536 exceeds 10% of free system memory.
W0000 00:00:1795513725.521099   84702 cpu_allocator_impl.cc:174] Allocation of 411713536 exceeds 10% of free sys

Epoch: 0 | Step: 100 | Loss: 10.674468994140625


W0000 00:00:1795514172.475142   84598 prefetch_dataset_op.cc:476] SEVERE STARVATION: Element UID 1795514144571712306 in buffer 'PrefetchDataset/_12' sat unconsumed for 27.9022 seconds!


Cancelling training.
